# Question 04: Giải mã "Chi phí ẩn" - Giá niêm yết có phải là giá thực tế?

**Câu hỏi nghiên cứu:**
Sinh viên thường chỉ nhìn vào giá thuê (Price) mà quên mất các chi phí sinh hoạt đi kèm (Điện, Nước, Rác, Dịch vụ...). Với dữ liệu đã được trích xuất chi tiết, chúng ta sẽ tính toán **Tổng Chi Phí Sở Hữu (TCO - Total Cost of Ownership)**.

**Công thức tính TCO:**
$$TCO = Price + (P_{dien} \times 100kWh) + P_{nuoc}^* + P_{rac} + P_{dichvu}$$

*Trong đó:*
- **Điện:** Giả định tiêu thụ trung bình 100 kWh/tháng.
- **Nước:**
  - Nếu đơn giá < 30.000đ $\rightarrow$ Tính theo khối (Giả định 4 khối/người).
  - Nếu đơn giá $\ge$ 30.000đ $\rightarrow$ Tính trọn gói theo đầu người.
- **Rác & Dịch vụ:** Cộng trực tiếp phí hàng tháng.

In [2]:
# --- Question 04: Tính toán Chi phí ẩn từ dữ liệu có sẵn ---

# 1. Chuẩn hóa dữ liệu số (Tránh lỗi string/object)
cols_utility = ['electric_price', 'water_price', 'garbage_price', 'service_price']
for col in cols_utility:
    # Chuyển về số, lỗi thành NaN
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 2. Xử lý giá trị thiếu (Missing Values Imputation)
# Nếu không có dữ liệu, dùng mức giá thị trường trung bình để ước tính TCO cho sát thực tế
df['electric_price_clean'] = df['electric_price'].fillna(3500)   # Mặc định 3.5k/kwh
df['garbage_price_clean'] = df['garbage_price'].fillna(20000)    # Mặc định 20k/tháng
df['service_price_clean'] = df['service_price'].fillna(0)        # Mặc định 0

# 3. Logic tính tiền NƯỚC (Phân biệt giá khối vs giá người)
def calc_water_cost(price):
    if pd.isna(price) or price == 0:
        return 100000 # Mặc định 100k/người nếu thiếu data
    if price < 30000:
        return price * 4 # Giá theo khối (Giả định dùng 4 khối)
    return price # Giá trọn gói (theo người/phòng)

df['water_cost_estimated'] = df['water_price'].apply(calc_water_cost)

# 4. Tính Tổng chi phí tiện ích (Utility Cost)
# Giả định: 1 người dùng 100 số điện
df['total_utility_cost'] = (
    (df['electric_price_clean'] * 100) + 
    df['water_cost_estimated'] + 
    df['garbage_price_clean'] + 
    df['service_price_clean']
)

# 5. Tính TCO và Tỷ lệ phí ẩn
df['total_cost_tco'] = df['price'] + df['total_utility_cost']
df['hidden_fee_ratio'] = (df['total_utility_cost'] / df['price']) * 100

# --- TRỰC QUAN HÓA KẾT QUẢ ---

# Biểu đồ 1: Scatter Plot so sánh Giá thuê vs TCO
plt.figure(figsize=(12, 6))
# Lọc bỏ nhiễu để vẽ cho đẹp (Giá < 20tr)
plot_data = df[(df['price'] > 0) & (df['price'] < 20000000)].sample(min(500, len(df)))

sns.scatterplot(x='price', y='total_cost_tco', data=plot_data, 
                color='orange', alpha=0.6, label='Giá Thực tế (TCO)')
# Đường tham chiếu y=x
plt.plot([0, 20e6], [0, 20e6], 'r--', label='Giá Niêm yết')

plt.title('Question 04: Thực tế sinh viên phải trả bao nhiêu mỗi tháng?', fontsize=14)
plt.xlabel('Giá thuê niêm yết (VND)')
plt.ylabel('Tổng chi phí hàng tháng (VND)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig('Q4_price_vs_tco.png')
plt.show()

# Biểu đồ 2: Phân phối các thành phần chi phí
avg_costs = pd.DataFrame({
    'Loại phí': ['Điện (100kWh)', 'Nước', 'Dịch vụ & Rác'],
    'Chi phí TB (VND)': [
        (df['electric_price_clean'].mean() * 100),
        df['water_cost_estimated'].mean(),
        (df['garbage_price_clean'].mean() + df['service_price_clean'].mean())
    ]
})

plt.figure(figsize=(8, 5))
sns.barplot(x='Loại phí', y='Chi phí TB (VND)', data=avg_costs, palette='Blues')
plt.title('Trung bình các khoản phí phát sinh hàng tháng', fontsize=12)
for i, v in enumerate(avg_costs['Chi phí TB (VND)']):
    plt.text(i, v + 5000, f"{v:,.0f}đ", ha='center', fontweight='bold')
plt.savefig('Q4_avg_utility_cost.png')
plt.show()

print(f"Insight: Trung bình mỗi tháng sinh viên phải đóng thêm {df['total_utility_cost'].mean():,.0f} VND ngoài tiền phòng.")

NameError: name 'df' is not defined